In [ ]:
import os
os.environ["AMDGPU_TARGETS"] = "gfx1032"
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "10.3.0"
import time
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import torchvision
from PIL import Image
import numpy as np
from dotenv import load_dotenv
import math 
import time
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler, Dataset, random_split
import cv2
import torch.nn as nn
import torch.optim as optim
from EAST import EAST
import torch.nn.functional as F
import re
torch.cuda.set_per_process_memory_fraction(0.5, 0)



In [ ]:
#controls

rboxOn = True
testcheck = False


In [ ]:
#class for the dataset
class imgDataset(Dataset):
    def __init__(self, path_imgs, path_labelAndcoords, box_shrink = 0.85):
        super().__init__()
        self.path_imgs = path_imgs
        self.path_labelAndCoords = path_labelAndcoords
        self.box_shrink = box_shrink

    def __len__(self):
        return len(self.path_imgs)
    
    def __getitem__(self, index):
        res = 512
        scale_factor = 4

        #img to tensor
        path_img = self.path_imgs[index]
        img = Image.open(path_img)
        transform = transforms.Compose([
            transforms.ColorJitter(
                brightness=0.2, 
                contrast=0.2, 
                saturation=0.2, 
                hue=0.1
            ),
            transforms.ToTensor(),

        ])
        img = transform(img)

        labelsForImg = []
        coordsForImg = []
        textAndCoord = self.path_labelAndCoords[index]

        #read the text file and parse the labels and coordinates
        with open(textAndCoord, 'r', encoding='utf-8-sig') as file: 
            line = file.readline()
            while line:
                line = line.strip()
                if line:
                    labelAndcoord = line.split(',')
                    labelsForImg.append([i for i in labelAndcoord[8]])
                    coords = [int(float(f) / 1) for f in labelAndcoord[:8]]
                    coordsForImg.append(coords)       
                line = file.readline() 
        # make coordmap for the image
        TrueMap = torch.zeros(1,img.shape[1], img.shape[2])
        if rboxOn:
            corners = torch.zeros(5,img.shape[1], img.shape[2])
        else:
            corners = torch.zeros(8,img.shape[1], img.shape[2])
            
        for box in coordsForImg:


            cx = (box[0] + box[2] + box[4] + box[6]) / 4
            cy = (box[1] + box[3] + box[5] + box[7]) / 4

            shrunkn_box = []
            for i in range(0,8,2):
                x_point = box[i]
                y_point = box[i+1]
                x_new = int(cx + self.box_shrink*(x_point - cx))
                y_new = int(cy + self.box_shrink*(y_point - cy))
                shrunkn_box.append(x_new)
                shrunkn_box.append(y_new)

            # parsing coords
            # gets egdes
            edges = [
                (shrunkn_box[0:2], shrunkn_box[2:4]), # (933,255) → (954,255)  top edge
                (shrunkn_box[2:4], shrunkn_box[4:6]),  # (954,255) → (956,277)  right edge
                (shrunkn_box[4:6], shrunkn_box[6:8]), # (956,277) → (936,277)  bottom edge
                (shrunkn_box[6:8], shrunkn_box[0:2])  # (936,277) → (933,255)  left edge
            ]

            #bounding box;
            #get coreners of box not boundning
            corners_xy = [(shrunkn_box[j], shrunkn_box[j+1]) for j in range(0, 8, 2)]
            min_x = min(shrunkn_box[0], shrunkn_box[2], shrunkn_box[4], shrunkn_box[6])
            max_x = max(shrunkn_box[0], shrunkn_box[2], shrunkn_box[4], shrunkn_box[6])
            min_y = min(shrunkn_box[1], shrunkn_box[3], shrunkn_box[5], shrunkn_box[7])
            max_y = max(shrunkn_box[1], shrunkn_box[3], shrunkn_box[5], shrunkn_box[7])

            x_delta = shrunkn_box[2] - shrunkn_box[4]
            y_delta = shrunkn_box[3] - shrunkn_box[5]
            if x_delta == 0 or y_delta == 0:
                rotate = 0
            else:
                rotate = math.atan(y_delta / x_delta)
            for x in range(min_x, max_x + 1):
                for y in range(min_y, max_y + 1):
                    cnt = 0
                    for edge in edges:
                        (x1,y1), (x2,y2) = edge
                        if (y < y1) != (y < y2) and y2 != y1 and x < x1 + (y - y1) / (y2 - y1) * (x2 - x1):
                            cnt += 1
                    if cnt % 2 == 1:
                        TrueMap[0][y][x] = 1
                    
                    if TrueMap[0][y][x] == 1 and False == rboxOn:  # only for pixels inside the box
                        for k, (cxt, cyt) in enumerate(corners_xy):
                            corners[k*2][y][x]   = (cxt - x) / res # ∆x to corner k
                            corners[k*2+1][y][x] = (cyt - y) / res # ∆y to corner k
                            # add quad version
                    else:
                        # rbox  channels # change to scale with img ?
                        
                        for k, edge in enumerate(edges):
                            if TrueMap[0][y][x] == 1:
                                (x1,y1),(x2,y2) = edge
                                top = abs(((x2-x1)*(y1-y))-((x1-x)*(y2-y1)))
                                bottom = math.sqrt((x2-x1)**2+(y2-y1)**2)
                                if top == 0 or bottom == 0:
                                    corners[k][y][x] = 0
                                else:
                                    corners[k][y][x] = (top/bottom) / res
                        corners[4][y][x] = rotate
                        
        angle = transforms.RandomRotation.get_params([-20, 20]) #float(torch.randint(1, 4, (1,)).item() * 90)


        width , height = crop_based_on_rotate(img, angle)

        #print(width , height)

        crop_transform = transforms.CenterCrop((height,width))

        img = TF.rotate(img, angle,expand=True)
        img = crop_transform(img)#TF.crop(img, top , left, height, width)
        img = img.unsqueeze(0)
        imga = F.interpolate(img, size=(int(res / scale_factor), int(res / scale_factor)), mode='bilinear', align_corners=False)
        imga = imga.squeeze(0)

        #TrueMap = TF.crop(img=TrueMap, top=top, left=left, height=h, width=w)
        TrueMap = TF.rotate(TrueMap, angle,expand=True)
        TrueMap = crop_transform(TrueMap)#TF.crop(TrueMap, top , left, height, width)
        TrueMap = TrueMap.unsqueeze(0)
        TrueMapa = F.interpolate(TrueMap, size=(int(res / scale_factor), int(res / scale_factor)), mode='bilinear', align_corners=False)
        mask = TrueMapa.squeeze(0).squeeze(0) > 0.5   # low-res mask, matches cornersa's H/W
        TrueMapa = TrueMapa.squeeze(0)

        #corners = TF.crop(img=corners, top=top, left=left, height=h, width=w)
        corners = TF.rotate(corners, angle,expand=True)
        corners = crop_transform(corners)#TF.crop(corners, top , left, height, width)
        corners = corners.unsqueeze(0)
        cornersa = F.interpolate(corners, size=(int(res / scale_factor), int(res / scale_factor)), mode='bilinear', align_corners=False)
        angle_rad = angle * (math.pi / 180)
        cornersa[0][4][mask] += angle_rad
        cornersa = cornersa.squeeze(0)
        
        return [imga, TrueMapa, cornersa]

def crop_based_on_rotate(tensor, angle_deg):

    '''
    moded based on https://stackoverflow.com/questions/5789239/calculate-largest-inscribed-rectangle-in-a-rotated-rectangle
    and https://stackoverflow.com/questions/16702966/rotate-image-and-crop-out-black-borders
    '''

    ang = math.radians(angle_deg)
    height_img , width_img = int(tensor.shape[1]), int(tensor.shape[2])

    quadrant = math.floor(ang / (math.pi / 2)) & 3

    if((quadrant & 1) == 0):
        sign_alpha = ang
    else:
        sign_alpha = math.pi - ang

    alpha = (sign_alpha % math.pi + math.pi) % math.pi

    alpha_sin , alpha_cos = math.sin(alpha), math.cos(alpha)

    bb = [
        width_img * alpha_cos + height_img * alpha_sin, #w
        width_img * alpha_sin + height_img * alpha_cos #h
    ]

    if(width_img < height_img):
        gamma = math.atan2(bb[0], bb[1])
    else:
        gamma = math.atan2(bb[1], bb[0])

    delta = math.pi - alpha - gamma

    if(width_img < height_img):
        length = height_img
    else:
        length = width_img
    
    d = length * alpha_cos
    a = d * alpha_sin / math.sin(delta)

    y = a * math.cos(gamma)
    x = y * math.tan(gamma)


    return round(bb[0] - 2 * x), round(bb[1] - 2 * y)


In [ ]:
#check the existence of the dataset and the labels. also check if the images and labels are in the same order
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]
def test_img_dataset():
    load_dotenv()
    directory_train = os.getenv('Directory_train')
    print(directory_train)
    directory_train_textAndCoords = os.getenv('directory_train_textAndCoords')
    print(directory_train_textAndCoords)
    # Generate paths for training images and labels
    train_img_paths = [
    os.path.join(directory_train, f) 
    for f in sorted(os.listdir(directory_train), key=natural_sort_key)
    ]
    #train_img_paths = [os.path.join(directory_train, f) for f in os.listdir(directory_train)]
    num = 0
    print(train_img_paths[num])
    #train_label_paths = [os.path.join(directory_train_textAndCoords, f) for f in os.listdir(directory_train_textAndCoords)]
    
    train_label_paths = [
        os.path.join(directory_train_textAndCoords, f) 
        for f in sorted(os.listdir(directory_train_textAndCoords), key=natural_sort_key)
    ]
    print(train_label_paths[num])
    #making objs
    train_dataset = [train_img_paths, train_label_paths]
    train_dataset = imgDataset(train_img_paths, train_label_paths)
    img, TrueMap, corners = train_dataset[num]

    print(f"shape: {img.shape}", type(img)) # Should be [1, 360, 360] since it's grayscale
    print(f"TrueMap: {TrueMap.shape}",type(TrueMap))
    print(f"corners:{corners.shape}",type(corners))
    return img, TrueMap, corners, train_dataset

if 1==1:
    img, TrueMap, corners, train_dataset = test_img_dataset()

In [ ]:
#loss functions
def balanced_cross_entropy_loss(preds, targets, epsilon=1e-3, pos_weight_mult=1.0) -> torch.Tensor:
    beta = 1 - torch.mean(targets.float())
    #beta = torch.clamp(beta * pos_weight_mult, max=0.99)
    #print(f"beta_after_clamp={beta.item():.4f}")
    preds = torch.clamp(preds, epsilon, 1.0 - epsilon)
    return ((-beta * targets * torch.log(preds)) - 
            (1-beta)*(1-targets)*(torch.log(1-preds))).mean()

#lg this is what is breaking
def quad_loss(preds, targets, TrueMap, epsilon=1e-3) -> torch.Tensor :
    mask = TrueMap
    loss = torch.nn.functional.smooth_l1_loss(preds * mask, targets * mask, reduction='sum')

    normalizer =(mask.sum() + epsilon) * 8
    return loss / ( normalizer)

def dice(pred: torch.Tensor, gt: torch.Tensor, TrueMap: torch.Tensor, lambda_theta=10) -> torch.Tensor:
    mask = TrueMap
    inter = 2*((pred * gt * mask).sum())
    union = ((pred * mask).sum() + (gt * mask).sum()) 
    loss = 1.0 - (inter/union)

    #print("loss",type(loss),loss.shape,loss)
    return loss

#four if i do geo as4 #

#four if i do geo as4 #
def rbox(pred: torch.Tensor, gt: torch.Tensor, TrueMap: torch.Tensor, lambda_theta=1, smooth: float = 1e-3) -> torch.Tensor:
    mask = TrueMap
    #die = dice(pred = pred,gt = gt,TrueMap = TrueMap,lambda_theta = lambda_theta) 
    #die *= 0.01
    #print(die)
    pred_m = pred * mask
    gt_m   = gt * mask
    '''with torch.no_grad():
        fg = mask.squeeze(1) > 0.5
        if fg.sum() > 0:
            print(f"gt_dist  min/mean/max: {gt_m[:, :4][fg.unsqueeze(1).expand(-1,4,-1,-1)].min():.4f} / "
                  f"{gt_m[:, :4][fg.unsqueeze(1).expand(-1,4,-1,-1)].mean():.4f} / "
                  f"{gt_m[:, :4][fg.unsqueeze(1).expand(-1,4,-1,-1)].max():.4f}")
            print(f"pred_dist min/mean/max: {pred_m[:, :4][fg.unsqueeze(1).expand(-1,4,-1,-1)].min():.4f} / "
                  f"{pred_m[:, :4][fg.unsqueeze(1).expand(-1,4,-1,-1)].mean():.4f} / "
                 f"{pred_m[:, :4][fg.unsqueeze(1).expand(-1,4,-1,-1)].max():.4f}")'''

    inter_h = torch.min(pred_m[:, 0], gt_m[:, 0]) + torch.min(pred_m[:, 2], gt_m[:, 2])
    #print("inter_h", inter_h.shape, inter_h.min(), inter_h.max())
    inter_w = torch.min(pred_m[:, 1], gt_m[:, 1]) + torch.min(pred_m[:, 3], gt_m[:, 3])
    #print("inter_w", inter_w.shape, inter_w.min(), inter_w.max())
    inter   = inter_h * inter_w 
    #print("inter", inter.shape, inter.min(), inter.max())

    pred_area = (pred_m[:, 0] + pred_m[:, 2]) * (pred_m[:, 1] + pred_m[:, 3])
    #print("pred_area", pred_area.shape, pred_area.min(), pred_area.max())
    gt_area = (gt_m[:, 0]   + gt_m[:, 2])   * (gt_m[:, 1]   + gt_m[:, 3])
    #print("gt_area", gt_area.shape, gt_area.min(), gt_area.max())
    union = ((pred_area + gt_area - inter) + smooth)
    #print("union", union.shape, union.min(), union.max())

    iou = (inter + smooth) / (union + smooth)
    #iou = torch.clamp(iou, min=1e-4, max=1.0)
    #print("iou", iou.shape, iou.min(), iou.max())
    
    L_aabb = -torch.log(iou)#.clamp(min=0.0, max=1.0)
    #print("L-aabb", L_aabb.shape, L_aabb.min(), L_aabb.max())

    L_theta = 1 - torch.cos(pred[:, 4] - gt[:, 4])
    #print("L_theta", L_theta.shape, L_theta.min(), L_theta.max())

    total_loss = (L_aabb + lambda_theta * L_theta).unsqueeze(dim=1)
    #print("total_loss", total_loss.shape, total_loss.min(), total_loss.max())

    masked_loss = total_loss * mask
    #print("masked_loss", masked_loss.sum(), masked_loss.shape)

    denom = mask.sum() + 1e-5
    #print("denom", denom)
    #print(f"L_aabb_mean: {(L_aabb.unsqueeze(1)*mask).sum()/denom:.4f}  |  L_theta_weighted_mean: {(lambda_theta*L_theta.unsqueeze(1)*mask).sum()/denom:.4f}| iou: {(iou*mask).sum()/denom:.4f}")
    return (masked_loss.sum() / denom )

In [ ]:
#test loss
def test_loss_funtions(img,TrueMap,corners):
    torch.cuda.empty_cache()
    model = EAST(color_channel=3,scale_factor=4)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    img = img.unsqueeze(0)
    img = img.to(device)
    TrueMap = TrueMap.unsqueeze(0)
    TrueMap = TrueMap.to(device)
    corners = corners.unsqueeze(0)
    print(corners.shape)
    print("TrueMap", TrueMap.shape)
    print("IMG",img.shape)
    corners = corners.to(device)

    model.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    optimizer.zero_grad()

    outputs = model(img)

    score_map , geo_map, quad_geo_map = outputs

    text_mask = (score_map >= 0.5).float().squeeze(dim=0)
    valid_scores = (score_map*text_mask).squeeze(dim = 0)
    print("valid_scores",valid_scores.shape,valid_scores.min(),valid_scores.max())
 
    a = balanced_cross_entropy_loss(preds=score_map,targets=TrueMap)
    #print("balanced_cross_entropy_loss: ", a)
    if rboxOn == True:
        c = 0.01 * rbox(pred = quad_geo_map, gt = corners, TrueMap = TrueMap)
        print("rbox_loss: ", c)
        print("a: ", a)
        d = a + c
        print("d: ", d)

    else:
        c = quad_loss(preds = geo_map, targets = corners, TrueMap = TrueMap)
        print("quad_loss: ", c)
    d = a + c
    #print("comp", d)
if testcheck:
    candidate_boxes = test_loss_funtions(img,TrueMap,corners)

In [ ]:
def visualize_rbox_edges(img, TrueMap, corners):
    img_np = img.squeeze().cpu().numpy()          # (3, H, W) for RGB
    if img_np.ndim == 3:
        img_np = np.transpose(img_np, (1, 2, 0))   # -> (H, W, 3)
    score_np   = TrueMap.squeeze().cpu().numpy()   # (H, W)
    corners_np = corners.cpu().numpy()             # (5, H, W) since rboxOn=True

    H, W = score_np.shape
    edge_names = ["Top Edge", "Right Edge", "Bottom Edge", "Left Edge"]

    panels = []

    for k in range(4):
        img_u8 = (img_np * 255).clip(0, 255).astype(np.uint8)
        img_resized = cv2.resize(img_u8, (W, H), interpolation=cv2.INTER_AREA)
        canvas = cv2.cvtColor(img_resized, cv2.COLOR_RGB2BGR)

        dist_map = corners_np[k]                   # (H, W) raw distances

        # only care about pixels inside the score map
        mask = score_np == 1
        if mask.sum() == 0:
            panels.append(canvas)
            continue

        dist_inside = dist_map[mask]
        d_min = dist_inside.min()
        d_max = dist_inside.max()
        d_range = d_max - d_min if d_max != d_min else 1.0

        # brightness = inverse distance (closer → brighter)
        heat = np.zeros((H, W), dtype=np.float32)
        heat[mask] = 1.0 - (dist_map[mask] - d_min) / d_range   # [0,1], 1=closest

        # map to colormap (INFERNO: dark=far, bright=close)
        heat_u8 = (heat * 255).astype(np.uint8)
        colored = cv2.applyColorMap(heat_u8, cv2.COLORMAP_INFERNO)

        # blend only inside-mask pixels onto the grayscale canvas
        mask_3ch = np.stack([mask]*3, axis=-1)
        blended = np.where(mask_3ch, colored, canvas)

        # label
        cv2.putText(blended, edge_names[k], (10, 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 255, 255), 2)

        panels.append(blended)

    # stack 2x2 grid
    top    = np.hstack([panels[0], panels[1]])
    bottom = np.hstack([panels[2], panels[3]])
    grid   = np.vstack([top, bottom])

    cv2.imshow("RBOX Edge Distance Heatmaps", grid)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
if 1==1:
    img, TrueMap, corners = train_dataset[0]   # fresh fetch, avoids stale/rerun state
    print(img.shape)
    print(TrueMap.shape)
    print(corners.shape)
    res = 512
    scale_factor = 1
    TrueMap_vis = F.interpolate(TrueMap.unsqueeze(0), size=(512, 512), mode='bilinear', align_corners=False)
    corners_vis = F.interpolate(corners.unsqueeze(0), size=(512, 512), mode='bilinear', align_corners=False)

    visualize_rbox_edges(img, TrueMap_vis.squeeze(0), corners_vis.squeeze(0))


In [ ]:
#training and validation cycles
def train_cycle(model, dataset_loaded, device, optimizer, rboxOn=True,accumulation_steps = 4):
    model.train()  # Make sure model is in train mode!
    start = time.time()
    running_loss = 0.0
    total_samples = 0
    optimizer.zero_grad()  # Zero gradients at the very start

    for i, (imgs, TrueMap, corners) in enumerate(dataset_loaded):
        batch_size = imgs.size(0)
        total_samples += batch_size
        imgs = imgs.to(device)
        TrueMap = TrueMap.to(device)
        corners = corners.to(device)
        score_map, geo_map, quad_geo_map = model(imgs)
        loss_score_map = balanced_cross_entropy_loss(preds=score_map, targets=TrueMap)
        if rboxOn:
            #print(quad_geo_map.shape)
            #print(corners.shape)
            #print(TrueMap.shape)
            loss_geo_map = rbox(pred=quad_geo_map, gt=corners, TrueMap=TrueMap)
        else:
            loss_geo_map = quad_loss(preds=geo_map, targets=corners, TrueMap=TrueMap)
        total_loss = loss_score_map + (0.01 * loss_geo_map)
        #print(f"score_loss={loss_score_map.item():.4f}  geo_loss={loss_geo_map.item():.4f}")
        running_loss += total_loss.item() * batch_size

        scaled_loss = total_loss / accumulation_steps
        scaled_loss.backward()

        if (i + 1) % accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            optimizer.zero_grad()

    if (i + 1) % accumulation_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        optimizer.zero_grad()

    # Calculate true average loss per sample
    avg_loss = running_loss / total_samples
    print(f"Avg Loss: {avg_loss:.4f} | Time taken: {time.time() - start:.2f} seconds ")
    return avg_loss

def val_cycle(model, dataset_loaded, device, rboxOn=True):

    model.eval()
    start = time.time()
    running_loss = 0.0
    total_iou = 0
    total_images = 0
    
    with torch.no_grad():
        for imgs, TrueMap, corners in dataset_loaded:

            imgs = imgs.to(device)
            batch_size = imgs.size(0)
            TrueMap = TrueMap.to(device)
            corners = corners.to(device)

            score_map, geo_map, quad_geo_map = model(imgs)

            loss_score_map = balanced_cross_entropy_loss(preds = score_map, targets = TrueMap)
            if rboxOn:
                loss_geo_map = rbox(pred = quad_geo_map, gt = corners, TrueMap = TrueMap)
            else:
                loss_geo_map = quad_loss(preds = geo_map, targets = corners, TrueMap = TrueMap)
            total_loss = loss_score_map + (1.0 * loss_geo_map)
            running_loss += total_loss.item() #* batch_size

            preds_binary = (score_map >= 0.5).float()
            
            for i in range(imgs.size(0)):
                intersection = torch.logical_and(preds_binary[i], TrueMap[i]).sum().item()
                union = torch.logical_or(preds_binary[i], TrueMap[i]).sum().item()
                iou = intersection / union if union > 0 else 1.0 
                total_iou += iou
            
            total_images += imgs.size(0)


    avg_loss = running_loss / total_images
    correct_avg = total_iou / total_images

    print(f"Time taken: {time.time() - start:.2f} seconds")
    print(f"Avg Loss: {avg_loss:.4f}")
    print(f"Avg Pixel IoU: {correct_avg:.4f}")
    return correct_avg, avg_loss

In [ ]:
#preprocessing for nms
def pre_nms_loacl(TrueMap, corners, threshold=0.8) -> torch.Tensor:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    TrueMap = TrueMap.to(device)
    corners = corners.to(device)
    valid_scores = TrueMap.to(device)
    #print("valid_scores.shape:", valid_scores.shape)
    #print("TrueMap.shape:", TrueMap.shape)
    #print("corners.shape:", corners.shape)

    top = corners[:, 0]
    right = corners[:, 1]
    bottom = corners[:, 2]
    left = corners[:, 3]
    valid_scores = valid_scores[:, 0]
    #print("valid_scores.shape:", valid_scores.shape)
    #print("bottom.shape:", bottom.shape)
    #print("left.shape:", left.shape)
    #print("right.shape:", right.shape)
    #print("top.shape:", top.shape)
    height, width = TrueMap.shape[-2], TrueMap.shape[-1]
    rows = torch.arange(height, device=device)
    cols = torch.arange(width, device=device)
    y, x = torch.meshgrid(rows, cols, indexing='ij')
    y = y.unsqueeze(dim=0).unsqueeze(dim=0)
    x = x.unsqueeze(dim=0).unsqueeze(dim=0)
    #print("x.shape:", x.shape)
    #print("y.shape:", y.shape)
    #print()
    x_min = (x - left * width) *TrueMap
    y_min = (y - top * height) *TrueMap
    x_max = (x + right * width) *TrueMap
    y_max = (y + bottom * height)*TrueMap
    x_min = torch.squeeze(x_min, dim=0)
    y_min = torch.squeeze(y_min, dim=0)
    x_max = torch.squeeze(x_max, dim=0)
    y_max = torch.squeeze(y_max, dim=0)
    x_min = torch.clamp(x_min, min=0, max=width)
    y_min = torch.clamp(y_min, min=0, max=height)
    x_max = torch.clamp(x_max, min=0, max=width)
    y_max = torch.clamp(y_max, min=0, max=height)
    #print("x_min.shape:", x_min.shape)
    #print("y_min.shape:", y_min.shape)
    #print("x_max.shape:", x_max.shape)
    #print("y_max.shape:", y_max.shape)
    #print("valid_scores.shape:",(valid_scores.shape))
    candidate_boxes = torch.stack([x_min, y_min, x_max, y_max, valid_scores])
    #adding shape 4
    candidate_boxes = candidate_boxes.permute(1, 0, 2, 3)
    return candidate_boxes

In [ ]:
#nms funtions
def iou_func(pred,truth):

    inter_h = torch.min(pred[0], truth[0]) + torch.min(pred[2], truth[2])

    #print("inter_h", inter_h)

    inter_w = torch.min(pred[1], truth[1]) + torch.min(pred[3], truth[3])

    #print("inter_w",inter_w)

    inter   = inter_h * inter_w 

    #print("inter",inter)

    pred_area = (pred[0] + pred[2]) * (pred[1] + pred[3])

    #print("pred_area",pred_area)

    gt_area = (truth[0]   + truth[2])   * (truth[1]   + truth[3])

    #print("gt_area",gt_area)

    union = pred_area + gt_area - inter

    #print("union",union)

    iou_fin = inter / union 

    #print("iou_fin",iou_fin)

    return iou_fin

def shouldMerge(p, g, threshold):
    iou = iou_func(p,g).mean()
    if iou < 0.0:
        iou = 0
    if iou >= threshold:
        return True
    return False


def weightedMerge(p, g):
    x_min_p, y_min_p, x_max_p, y_max_p, scores_p = p[0], p[1], p[2], p[3], p[4]
    x_min_g, y_min_g, x_max_g, y_max_g, scores_g = g[0], g[1], g[2], g[3], g[4]

    new_scores = scores_p + scores_g + 1e-8
    x_min_ult = ((scores_p * x_min_p) + (scores_g * x_min_g)) / new_scores
    y_min_ult = ((scores_p * y_min_p) + (scores_g * y_min_g)) / new_scores
    x_max_ult = ((scores_p * x_max_p) + (scores_g * x_max_g)) / new_scores
    y_max_ult = ((scores_p * y_max_p) + (scores_g * y_max_g)) / new_scores
    scores_ult = new_scores

    pixelmerged = torch.stack([x_min_ult, y_min_ult, x_max_ult, y_max_ult, scores_ult])

    pixelmerged_clamped = torch.clamp(pixelmerged, max=100)

    return pixelmerged_clamped


def nmsLocality(geometries, threshold, score_thresh=1e-3):
    #print("Input geometries shape:", geometries.shape)
    """
    geometries: [1, 5, H, W]  (x_min, y_min, x_max, y_max, score) per pixel
    """
    geometries = geometries.squeeze(0)          # [5, H, W]
    pred_boxes = []
    edge_sliced = torch.split(geometries, 1, dim=1)   # split along H -> rows

    #print("Edge sliced shape:", [r.shape for r in edge_sliced])
    for row in edge_sliced:
        row = row.squeeze(1)                     # [5, W]
        current_box = None
        current_row = []
        #print("Processing row shape:", row.shape)

        for pixel in torch.split(row, 1, dim=1):  # [5, 1]
            #print("Processing pixel shape:", pixel.shape)
            score = pixel[4].item()

            if score <= score_thresh:
                # background pixel: flush whatever we were building, skip it
                if current_box is not None:
                    current_row.append(current_box)
                    current_box = None
                continue

            if current_box is None or torch.all(current_box == 0):
                current_box = pixel
            elif shouldMerge(pixel, current_box, threshold):
                current_box = weightedMerge(pixel, current_box)
            else:
                current_row.append(current_box)
                current_box = pixel   # keep the pixel that broke the run

        if current_box is not None:               # flush end of row
            current_row.append(current_box)

        pred_boxes.append(current_row)

    tensor_list_pred_boxes = [torch.stack(row) for row in pred_boxes if len(row) > 0]
    if len(tensor_list_pred_boxes) == 0:
        return torch.zeros((1, 5, 1), dtype=geometries.dtype, device=geometries.device)
 
    return torch.cat(tensor_list_pred_boxes)
    


def standardNms(S, threshold):
    S = S.squeeze(dim=2)
    new_S = []
    for pixel in S:
        new_pixel = [layer.item() for layer in pixel]
        new_S.append(torch.tensor(new_pixel))
    new_S = torch.stack(new_S)

    x_and_ys = new_S[:, 0:4]
    scores = new_S[:, 4]
    v = torchvision.ops.nms(x_and_ys, scores, threshold)
    return v


In [ ]:
#test loss functions and nms
def test_loss_funtions(TrueMap, corners, threshold=0.8):


    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    TrueMap = TrueMap.unsqueeze(0).to(device)
    corners = corners.unsqueeze(0).to(device)
    print("corners", corners.shape)

    # use TrueMap itself as the "score" — 1.0 wherever a box exists, 0 elsewhere
    valid_scores = TrueMap.squeeze(0)
    print("valid_scores", valid_scores.shape, valid_scores.min(), valid_scores.max())

    # ground-truth corner channels: top, right, bottom, left
    top = corners[:, 0]
    right = corners[:, 1]
    bottom = corners[:, 2]
    left = corners[:, 3]

    fg_mask = (TrueMap[0,0] > 0.5)
    r, c = fg_mask.nonzero()[0].tolist()
    print("sample pixel:", r, c)
    print("top,right,bottom,left:", top[0,r,c].item(), right[0,r,c].item(), bottom[0,r,c].item(), left[0,r,c].item())

    # check how many of its immediate neighbors are also foreground
    neighbors = [(r-1,c),(r+1,c),(r,c-1),(r,c+1)]
    for nr, nc in neighbors:
        if 0 <= nr < 128 and 0 <= nc < 128:
            print(f"neighbor ({nr},{nc}) foreground:", fg_mask[nr,nc].item())


    print("top", top.shape, "right", right.shape, "bottom", bottom.shape, "left", left.shape)
    print("top_max", top.max(), "right_max", right.max(), "bottom_max", bottom.max(), "left_max", left.max())
    print("top_min",top.min(), "right_min",right.min(), "bottom_min",bottom.min(), "left_min",left.min())
    height, width = TrueMap.shape[-2], TrueMap.shape[-1]
    rows = torch.arange(height, device=device)
    cols = torch.arange(width, device=device)
    y, x = torch.meshgrid(rows, cols, indexing='ij')
    print("x",x.max(), "y",y.max())
    print("x",x.min(), "y",y.min())
    y = y.unsqueeze(dim=0).unsqueeze(dim=0)
    x = x.unsqueeze(dim=0).unsqueeze(dim=0)
    x_min = (x - left * width) *TrueMap
    y_min = (y - top * height) *TrueMap
    x_max = (x + right * width) *TrueMap
    y_max = (y + bottom * height)*TrueMap
    print("x_min_max",x_min.max(), "y_min_max",y_min.max(), "x_max_max",x_max.max(), "y_max_max",y_max.max())
    print("x_min_min",x_min.min(), "y_min_min",y_min.min(), "x_max_min",x_max.min(), "y_max_min",y_max.min())
    x_min = torch.squeeze(x_min, dim=0)
    y_min = torch.squeeze(y_min, dim=0)
    x_max = torch.squeeze(x_max, dim=0)
    y_max = torch.squeeze(y_max, dim=0)
    x_min = torch.clamp(x_min, min=0, max=width)
    y_min = torch.clamp(y_min, min=0, max=height)
    x_max = torch.clamp(x_max, min=0, max=width)
    y_max = torch.clamp(y_max, min=0, max=height)

    print("x_min", x_min.shape, "y_min", y_min.shape, "x_max", x_max.shape, "y_max", y_max.shape)
   # x_min, y_min, x_max, y_max = top, right, bottom, left

    candidate_boxes = torch.stack([x_min, y_min, x_max, y_max, valid_scores])

    #candidate_boxes = torch.stack([x_min, y_min, x_max, y_max, valid_scores])  # [5, 1, H, W]
    cb = candidate_boxes[:, 0]   # [5, H, W]  <-- FIXED: select batch, keep channels
    p1 = cb[:, r, c]             # use the same r, c from your foreground check above
    p2 = cb[:, r, c+1]
    print("box1:", p1.tolist())
    print("box2:", p2.tolist())

    ax1, ay1, ax2, ay2 = p1[:4]
    bx1, by1, bx2, by2 = p2[:4]
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0., ix2 - ix1), max(0., iy2 - iy1)
    inter = iw * ih
    area1 = max(0., ax2 - ax1) * max(0., ay2 - ay1)
    area2 = max(0., bx2 - bx1) * max(0., by2 - by1)
    union = area1 + area2 - inter
    print("IoU:", (inter / union).item() if union > 0 else 0)
    candidate_boxes = candidate_boxes.permute(1, 0, 2, 3)
    print("candidate_boxes", candidate_boxes.shape)


    merged = nmsLocality(candidate_boxes, 0.8)
    print("merged boxes after nmsLocality:", merged.shape)

    final = standardNms(merged, 0.5)
    print("final box count:", final)
    return merged, #final
    #need to fix so that edge is 1 and the far est is near zero

if testcheck:
    img, TrueMap, corners = train_dataset[0]
    merged_boxes = test_loss_funtions(TrueMap, corners)

In [ ]:
#second test loss functions and nms 
def test_loss_funtions_two(TrueMap, corners, threshold=0.8):

    print(TrueMap.shape)
    print(corners.shape)
    candidate_boxes = pre_nms_loacl (TrueMap, corners, threshold=0.8)
    print("candidate_boxes", candidate_boxes.shape)
    
    merged = nmsLocality(candidate_boxes, 0.8)
    print("merged boxes after nmsLocality:", merged.shape)
    print("merged type",type(merged))

    final = standardNms(merged, 0.5)
    print("final type",type(final))

    #merged_l = merged.tolist()
    #final_l = final.tolist()


    #candidate_boxes_truth_fin_fin = []
    #for box in final:
    #    index = box.item()
    #    candidate_boxes_truth_fin_fin.append(merged[index])
    candidate_boxes_truth_fin_fin = merged[final.flatten()] 
    print("final box count:",candidate_boxes_truth_fin_fin)
    print("final box count len:",len(candidate_boxes_truth_fin_fin))
    return merged, 

if 1==0:
    img, TrueMap, corners = train_dataset[0]
    merged_boxes = test_loss_funtions_two(TrueMap, corners)

In [ ]:
#thrid test loss functions and nms
def val_test(img,TrueMap,corners,model,device): 
    with torch.no_grad():
        #to decive
        print("Starting validation/test...")
        print("Image shape: ", img.shape)
        print("TrueMap shape: ", TrueMap.shape)
        print("Corners shape: ", corners.shape)
        img = img.unsqueeze(0).to(device)
        TrueMap = TrueMap.unsqueeze(0).to(device)
        corners = corners.unsqueeze(0).to(device)
        print("Image after unsqueeze: ", img.shape)
        print("TrueMap after unsqueeze: ", TrueMap.shape)
        print("Corners after unsqueeze: ", corners.shape)

        # the pred
        score_map, geo_map, quad_geo_map = model(img)

        print("Score map shape: ", score_map.shape)
        print("Geo map shape: ", geo_map.shape)
        print("Quad geo map shape: ", quad_geo_map.shape)

        #losses
        loss_score_map = balanced_cross_entropy_loss(preds = score_map, targets = TrueMap)
        if rboxOn:
            loss_geo_map = rbox(pred = quad_geo_map, gt = corners, TrueMap = TrueMap)
        else:
            loss_geo_map = quad_loss(preds = geo_map, targets = corners, TrueMap = TrueMap)
        total_loss = loss_score_map + (1.0 * loss_geo_map)
        
        match_percent = ((score_map >= 0.5).float() == TrueMap).float().mean().item()

        threshold = 0.2

        candidate_boxes_truth = pre_nms_loacl(TrueMap, corners, threshold=threshold)
        candidate_boxes_truth_merged = nmsLocality(candidate_boxes_truth, threshold)
        candidate_boxes_truth_fin = standardNms(candidate_boxes_truth_merged , threshold)

        candidate_boxes_truth_fin_fin = candidate_boxes_truth_merged[candidate_boxes_truth_fin.flatten()] 

        #candidate_boxes_truth_fin_fin = candidate_boxes_truth_fin_fin.permute(2,1,0)
        

        #print("candidate_boxes_truth_fin_fin",len(candidate_boxes_truth_fin_fin),candidate_boxes_truth_fin_fin.shape, candidate_boxes_truth_fin_fin)


        candidate_boxes_pred = pre_nms_loacl(TrueMap, quad_geo_map, threshold=threshold)
        candidate_boxes_pred_merged = nmsLocality(candidate_boxes_pred, threshold)
        #print(candidate_boxes_pred_merged)
        candidate_boxes_pred_fin = standardNms(candidate_boxes_pred_merged, threshold)

        candidate_boxes_pred_fin_fin = candidate_boxes_pred_merged[candidate_boxes_pred_fin.flatten()] 

        #candidate_boxes_pred_fin_fin = candidate_boxes_pred_fin_fin.permute(2,1,0)

        #print("candidate_boxes_pred_fin_fin",len(candidate_boxes_pred_fin_fin),candidate_boxes_pred_fin_fin.shape,candidate_boxes_pred_fin_fin)



        if len(candidate_boxes_pred_fin_fin) > len(candidate_boxes_truth_fin_fin):
            for i in range(len(candidate_boxes_pred_fin_fin) - len(candidate_boxes_truth_fin_fin)):
                zero_box = torch.zeros(1, 5, 1).to(device) 
                candidate_boxes_truth_fin_fin = torch.cat(
                    [candidate_boxes_truth_fin_fin, zero_box], dim=0
                )
        else:
            for i in  range(len(candidate_boxes_truth_fin_fin) - len(candidate_boxes_pred_fin_fin)):
                                zero_box = torch.zeros(1, 5, 1).to(device) 
                                candidate_boxes_pred_fin_fin = torch.cat(
                                    [candidate_boxes_pred_fin_fin, zero_box], dim=0
                                 )

        #print("candidate_boxes_truth_fin_fin_p",len(candidate_boxes_truth_fin_fin),candidate_boxes_truth_fin_fin.shape)
        #print("candidate_boxes_pred_fin_fin_p",len(candidate_boxes_pred_fin_fin), candidate_boxes_pred_fin_fin.shape)

        list_of_compared = []
        count = 0

        for i in range(0,len(candidate_boxes_truth_fin_fin)):

            #first layer auto removes
            best_a_like = torch.zeros(5,1,1).to(device)
            best_iou = torch.zeros(1).to(device)
            truth = candidate_boxes_truth_fin_fin[i]

            for j in range(i,len(candidate_boxes_pred_fin_fin)):

                pred = candidate_boxes_pred_fin_fin[j]
                iou = iou_func(pred,truth)
                #print(iou.shape,iou)

                if iou.item() >= best_iou.item():
                    best_a_like = pred
                    best_iou = iou

            iou = iou_func(best_a_like,truth)
            if iou.item() > 0:
                list_of_compared.append(float(iou))
            else:
                 list_of_compared.append(0)
            count = count + 1
        print("count: ",count,"  list_of_compared: ", list_of_compared)
        #list_of_compared = torch.tensor(list_of_compared)
        #print(list_of_compared.shape)
        acc = torch.tensor([float(x) for x in list_of_compared], dtype=torch.float).mean().item()
        #print(candidate_boxes_truth_fin_fin)
        #print(candidate_boxes_pred_fin_fin)
        print("rbox acc: ", acc)
        print("scoremap acc", match_percent)
        
             
                        
if 1==1:
     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
     model = EAST(color_channel=1, scale_factor=4)
     model.Initialize_weights()
     model.to(device)
     img, TrueMap, corners =    train_dataset[0]
     val_test(img,TrueMap,corners,model,device)

In [ ]:
#new validation function with nms and accuracy calculation

def new_val(model, dataset_loaded, device, rboxOn=True):
    model.eval()
    start = time.time()
    running_loss = 0.0
    total_images = 0
    total_acc_nms = 0
    total_acc_heatmap = 0
    threshold = 0.8
    threshold_maps = 0.5
    with torch.no_grad():
        for imgs, TrueMap, corners in dataset_loaded:
            #to decive
            imgs = imgs.to(device)
            batch_size = imgs.size(0)
            TrueMap = TrueMap.to(device)
            corners = corners.to(device)
    
            # the pred
            score_map, geo_map, quad_geo_map = model(imgs)

            #losses
            loss_score_map = balanced_cross_entropy_loss(preds = score_map, targets = TrueMap)
            if rboxOn:
                loss_geo_map = rbox(pred = quad_geo_map, gt = corners, TrueMap = TrueMap)
            else:
                loss_geo_map = quad_loss(preds = geo_map, targets = corners, TrueMap = TrueMap)
            total_loss = loss_score_map + (0.01 * loss_geo_map) 
            running_loss += total_loss.item() * batch_size

            TrueMap_binary = (TrueMap >= threshold_maps).float()
            total_acc_heatmap += ((score_map >= threshold_maps).float() == TrueMap_binary).float().mean().item() * batch_size
            #funny = ((quad_geo_map*TrueMap).float() == corners*TrueMap).float().mean()

            for i in range(batch_size):
                TrueMap_one = TrueMap[i]
                corners_one = corners[i]
                # Process each image in the batch
                candidate_boxes_truth = pre_nms_loacl(TrueMap_one.unsqueeze(0), corners_one.unsqueeze(0), threshold=threshold)
                candidate_boxes_truth_merged = nmsLocality(candidate_boxes_truth, threshold)
                candidate_boxes_truth_fin = standardNms(candidate_boxes_truth_merged , threshold)
                candidate_boxes_truth_fin_fin = candidate_boxes_truth_merged[candidate_boxes_truth_fin.flatten()] 
    
                #candidate_boxes_truth_fin_fin = candidate_boxes_truth_fin_fin.permute(2,1,0)
                #print("candidate_boxes_truth_fin_fin",len(candidate_boxes_truth_fin_fin),candidate_boxes_truth_fin_fin.shape)
    
                quad_geo_map_one = quad_geo_map[i]
                candidate_boxes_pred = pre_nms_loacl(TrueMap_one.unsqueeze(0), quad_geo_map_one.unsqueeze(0), threshold=threshold)
                candidate_boxes_pred_merged = nmsLocality(candidate_boxes_pred, threshold)
                candidate_boxes_pred_fin = standardNms(candidate_boxes_pred_merged, threshold)
                candidate_boxes_pred_fin_fin = candidate_boxes_pred_merged[candidate_boxes_pred_fin.flatten()] 
    
                #candidate_boxes_pred_fin_fin = candidate_boxes_pred_fin_fin.permute(2,1,0)
                #print("candidate_boxes_pred_fin_fin",len(candidate_boxes_pred_fin_fin),candidate_boxes_pred_fin_fin.shape)
    
    
    
                if len(candidate_boxes_pred_fin_fin) > len(candidate_boxes_truth_fin_fin):
                    for i in range(len(candidate_boxes_pred_fin_fin) - len(candidate_boxes_truth_fin_fin)):
                        zero_box = torch.zeros(1, 5, 1).to(device) 
                        candidate_boxes_truth_fin_fin = torch.cat(
                            [candidate_boxes_truth_fin_fin, zero_box], dim=0
                        )
                else:
                    for i in  range(len(candidate_boxes_truth_fin_fin) - len(candidate_boxes_pred_fin_fin)):
                                        zero_box = torch.zeros(1, 5, 1).to(device) 
                                        candidate_boxes_pred_fin_fin = torch.cat(
                                            [candidate_boxes_pred_fin_fin, zero_box], dim=0
                                        )
    
                #print("candidate_boxes_truth_fin_fin_p",len(candidate_boxes_truth_fin_fin),candidate_boxes_truth_fin_fin.shape)
                #print("candidate_boxes_pred_fin_fin_p",len(candidate_boxes_pred_fin_fin), candidate_boxes_pred_fin_fin.shape)
    
                list_of_compared = []
                count = 0
    
                for i in range(0,len(candidate_boxes_truth_fin_fin)):
        
                    #first layer auto removes
                    best_a_like = torch.zeros(5,1,1).to(device)
                    best_iou = torch.zeros(1).to(device)
                    truth = candidate_boxes_truth_fin_fin[i]
        
                    for j in range(i,len(candidate_boxes_pred_fin_fin)):
                        pred = candidate_boxes_pred_fin_fin[j]
                        iou = iou_func(pred,truth)
        
                        if iou.item() >= best_iou.item():
                            best_a_like = pred
                            best_iou = iou

                    '''tx1, ty1, tx2, ty2 = truth[0], truth[1], truth[2], truth[3]
                    px1, py1, px2, py2 = best_a_like[0], best_a_like[1], best_a_like[2], best_a_like[3]
                    ix1, iy1 = torch.max(tx1, px1), torch.max(ty1, py1)
                    ix2, iy2 = torch.min(tx2, px2), torch.min(ty2, py2)
                    inter = torch.clamp(ix2 - ix1, min=0) * torch.clamp(iy2 - iy1, min=0)
                    area_t = torch.clamp(tx2 - tx1, min=0) * torch.clamp(ty2 - ty1, min=0)
                    area_p = torch.clamp(px2 - px1, min=0) * torch.clamp(py2 - py1, min=0)
                    iou = (inter / (area_t + area_p - inter + 1e-8)).item()'''
                    iou = iou_func(best_a_like,truth)
                    if iou.item() > 0:
                        list_of_compared.append(float(iou))
                    else:
                        list_of_compared.append(0)
                    count = count + 1

                acc = torch.tensor([float(x) for x in list_of_compared], dtype=torch.float).mean().item()
                #print("count: ",count,"  list_of_compared: ", list_of_compared)
                #print(list_of_compared.shape)
                #print("acc for this image: ", acc)
                total_acc_nms += acc #/ 100  # Normalize 
                total_images += 1

    if total_images <= 0:
        avg_loss = 0.0
        avg_acc_nms = 0.0
        avg_acc_heatmap = 0.0
    else:
        avg_loss = running_loss / total_images 
        avg_acc_nms = total_acc_nms / total_images
        avg_acc_heatmap = total_acc_heatmap / total_images

    print(f"Total images processed: {total_images } | Time taken: {time.time() - start:.2f} seconds | Avg Loss: {avg_loss:.4f} | Avg Heatmap Accuracy: {avg_acc_heatmap:.4f}")
    return avg_acc_nms, avg_loss


In [ ]:
#Training function for the model
def train_model(model, loader_train, loader_val, scheduler, optimizer, cycles, midwaychanage = 55):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(device)
    model.to(device)

    best_val_loss  = float('inf')
    best_model_path = "east_model.pth"

    for cycle in range(cycles):
        print(f"Cycle: {cycle+1}/{cycles}")

        if(midwaychanage < cycle):
            optimizer.param_groups[0]['weight_decay'] = 1e-5

        model.train()
        train_loss = train_cycle(model, dataset_loaded=loader_train, device=device, optimizer=optimizer, accumulation_steps=2)

        model.eval()
        val_acc, val_loss = new_val(model, dataset_loaded=loader_val, device=device)

        scheduler.step()

        print(f"coutputs|| LR: {scheduler.get_last_lr()} |Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val IoU: {val_acc:.4f} | weight_decay {optimizer.param_groups[0]['weight_decay']}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            print(f"(------ Model improved — Val Loss: {best_val_loss:.4f}, saving. -----)")
            torch.save(model.state_dict(), best_model_path)

    return best_model_path


In [ ]:
#loads in paths
load_dotenv()
directory_train = os.getenv('Directory_train')
directory_train_textAndCoords = os.getenv('directory_train_textAndCoords')

In [ ]:
# Define path_imgs and path_labels
path_imgs = sorted(os.listdir(directory_train))
path_labels = sorted(os.listdir(directory_train_textAndCoords))

# Generate paths for training images and labels
test_img_paths = [
    os.path.join(directory_train, f) 
    for f in sorted(os.listdir(directory_train), key=natural_sort_key)
]
train_label_paths = [
        os.path.join(directory_train_textAndCoords, f) 
        for f in sorted(os.listdir(directory_train_textAndCoords), key=natural_sort_key)
]

In [ ]:
#making objs
train_dataset = [test_img_paths, train_label_paths]
train_dataset = imgDataset(test_img_paths, train_label_paths)
img, TrueMap, corners = train_dataset[6]
print(test_img_paths[6])
print(train_label_paths[6])
print(f"Image shape: {img.shape}", type(img)) # Should be [1, 360, 360] since it's grayscale
print(f"First TrueMap: {TrueMap.shape}",type(TrueMap))
print(f"First corners: {corners.shape}",type(corners))

#batch = custom_collate(train_dataset)


In [ ]:
#spltining the dataset into training and validation sets
val_split = 0.2
training_size = int((1 - val_split) * len(train_dataset))
val_size = len(train_dataset) - training_size
training_dataset, val_dataset = random_split(train_dataset, [training_size, val_size])
loader_train = DataLoader(training_dataset, batch_size=12, shuffle=True,num_workers=12, pin_memory=True)
loader_val = DataLoader(val_dataset, batch_size=10, shuffle=True,num_workers=8, pin_memory=True)

In [ ]:
#loading model
model = EAST(color_channel=3, scale_factor=4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
model.Initialize_weights()
model.to(device)


In [ ]:
# Loading model if it exists
load_path = os.getenv('Load_model')

if load_path and os.path.isfile(load_path):
    model.load_state_dict(torch.load(load_path))
    print(f"Model loaded successfully from: {load_path}")
else:
    print("Model not loaded, starting from scratch")

In [ ]:
#defualt model otipions
optimizer = optim.Adam(model.parameters(), lr = 0.001, weight_decay= 1e-4)
print(f"weight decay {optimizer.param_groups[0]['weight_decay']}")
#count = 0
'''for group in optimizer.param_groups:
            group['weight_decay'] = 1e-6
            count =+ 1
print(count)
print(f"weight decay new {optimizer.param_groups[0]['weight_decay']}")'''
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size= 50, gamma=0.1)
cycles = 3
midwaychanage = 50

In [ ]:
train_model(model = model, loader_train = loader_train, loader_val = loader_val,scheduler=scheduler, optimizer = optimizer, cycles = cycles, midwaychanage = midwaychanage)

In [ ]:
model_save_path = os.getenv('Model_save_path', 'east_model_last.pth')
torch.save(model.state_dict(), model_save_path)